# Fraud detection: understand and reproduce the evidence

Use the project Python 3.11 environment. These cells inspect verified data and frozen reports. They do not tune against the final test period. BankSim is a research simulation; ULB is anonymized historical card data.

In [ ]:
# Locate the repository, import reusable project functions, and explicitly choose a dataset.
# Loading verified data checks its manifest; this notebook does not train silently.
from pathlib import Path
import sys, json
root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "backend" / "src").is_dir())
sys.path.insert(0, str(root / "backend"))
import pandas as pd
import numpy as np
from src.data_loader import load_transaction_data
from src.features import build_features
from src.preprocessing import chronological_split
profile = "banksim"  # change to credit_card and rerun
frame = load_transaction_data(profile=profile)
frame.head()

## 1. Verify provenance and class imbalance

Predict the majority-class accuracy before running the next cell. Would that model catch fraud?

In [ ]:
# Inspect original source identity and rare-fraud prevalence before trusting accuracy.
# The all-legitimate baseline demonstrates why accuracy alone can mislead.
manifest = json.loads((root / "backend/data/raw" / f"{profile}.manifest.json").read_text())
print(manifest)
target = "fraud" if profile == "banksim" else "Class"
print(frame[target].value_counts())
print("Always-legitimate accuracy:", 1 - frame[target].mean())
frame.describe(include="all").T

## 2. Behavioural features without future information

For BankSim, a whole day is the smallest trustworthy ordering unit. Same-day rows cannot inform one another. IDs look up histories but are not classifier inputs.

In [ ]:
# Use exactly the feature builder shared by training and live scoring; labels are ignored.
features = build_features(frame, profile)
features.head()

In [ ]:
# Inspect one customer’s observations beside derived history, or explain PCA limits.
# Same-day payments cannot inform one another because within-day order is unknown.
if profile == "banksim":
    customer = frame.customer.value_counts().index[0]
    observations = frame[frame.customer == customer].head(12)
    display(observations)
    display(features.loc[observations.index])
else:
    print("PCA variables have no published raw business interpretation.")

## 3. Fit, calibrate, validate, test

Imputation/scaling and SMOTE learn only from the fitting block. Calibration retains original class prevalence; validation selects the model and threshold. Never resample the final test period.

In [ ]:
# Keep five complete chronological time groups: fit, calibrate, router train, validate, test.
# Group percentages describe unique times rather than exact row percentages.
parts = chronological_split(frame.drop_duplicates().reset_index(drop=True), profile)
{k: len(v) for k,v in vars(parts).items()}

## 4. Compare the models you claimed in your CV

Run scripts/python -m src.train --profile all before this cell. Compare the SMOTE candidate with the same algorithm without resampling. Do not assume resampling always helps.

In [ ]:
# Follow the current pointer to the matching frozen report; do not retrain to read results.
# Compare all candidates using metrics from the same run and test period.
pointer = json.loads((root / "backend/models/v2" / f"{profile}.current.json").read_text())
report = json.loads((root / "backend/reports/v2" / pointer["report"]).read_text())
comparison = pd.DataFrame({name: {key: value["test"][key] for key in ["accuracy","precision","recall","f1_score","average_precision","pr_auc","roc_auc"]} for name,value in report["models"].items()}).T
display(comparison)
print("Selected on validation:", report["selected_model"])
print("Threshold:", report["decision_threshold"])
# Auto may correctly keep an individual when the learned controller loses.
print("Auto policy:", report.get("auto_policy", {}).get("policy"))
print("Auto reason:", report.get("auto_policy", {}).get("reason"))
display(pd.Series(report.get("router", {}).get("validation_routes", {}), name="Validation route count"))


## 5. Explain mistakes and evaluation limits

Accuracy alone is insufficient. Read the confusion matrix as rows = actual [legitimate, fraud], columns = predicted [legitimate, fraud]. Global importance does not explain a specific transaction.

In [ ]:
# Read actual confusion counts and validation importance; neither is a causal diagnosis.
selected = report["models"][report["selected_model"]]["test"]
display(pd.DataFrame(selected["confusion_matrix"], index=["Actual legitimate", "Actual fraud"], columns=["Predicted legitimate", "Predicted fraud"]))
display(pd.DataFrame(report["feature_importance"]).head(10))
print("Quality goals:", report["quality_goals_met"])

## 6. Investigation exercise

Explain what a false positive costs and what a false negative costs. Inspect the saved PR curve points to understand the trade-off, but do not retune on this test set. For a new experiment, use the fitting/calibration/validation data and reserve a new future holdout for final evaluation.

Use the dashboard CSV workspace to score your own compatible transactions. Those uploaded-data results remain separate from this benchmark.